In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
jangedoo_utkface_new_path = kagglehub.dataset_download('jangedoo/utkface-new')

print('Data source import complete.')


Using Colab cache for faster access to the 'utkface-new' dataset.
Data source import complete.


In [4]:
jangedoo_utkface_new_path

'/kaggle/input/utkface-new'

In [5]:
base_path=jangedoo_utkface_new_path+"/UTKFace"

In [6]:
import os
import pandas as pd

data = []

for file in os.listdir(base_path):

    if file.endswith(".jpg"):
        parts = file.split("_")

        age = int(parts[0])
        gender = int(parts[1])

        filepath = os.path.join(base_path, file)

        data.append({
            "file_path": filepath,
            "age": age,
            "gender": gender
        })

df = pd.DataFrame(data)



In [7]:
df

,file_path,age,gender
0,/kaggle/input/utkface-new/UTKFace/26_0_2_20170...,26,0
1,/kaggle/input/utkface-new/UTKFace/22_1_1_20170...,22,1
2,/kaggle/input/utkface-new/UTKFace/21_1_3_20170...,21,1
3,/kaggle/input/utkface-new/UTKFace/28_0_0_20170...,28,0
4,/kaggle/input/utkface-new/UTKFace/17_1_4_20170...,17,1
...,...,...,...
23703,/kaggle/input/utkface-new/UTKFace/41_0_0_20170...,41,0
23704,/kaggle/input/utkface-new/UTKFace/42_0_0_20170...,42,0
23705,/kaggle/input/utkface-new/UTKFace/2_0_2_201612...,2,0
23706,/kaggle/input/utkface-new/UTKFace/65_1_0_20170...,65,1


In [ ]:
import os
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# -------------------------------------------------------
# 1. Dataset Path
# -------------------------------------------------------


# -------------------------------------------------------
# 2. Create DataFrame
# -------------------------------------------------------

data = []

for file in os.listdir(base_path):

    if file.endswith(".jpg"):
        parts = file.split("_")

        age = int(parts[0])
        gender = int(parts[1])

        filepath = os.path.join(base_path, file)

        data.append({
            "file_path": filepath,
            "age": age,
            "gender": gender
        })

df = pd.DataFrame(data)

print("Total Images:", len(df))
print(df.head())

# -------------------------------------------------------
# 3. Convert Gender to String (Required for binary mode)
# -------------------------------------------------------

df["gender"] = df["gender"].map({0:"male",1:"female"})

# -------------------------------------------------------
# 4. Train / Validation Split
# -------------------------------------------------------

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# -------------------------------------------------------
# 5. Image Generators
# -------------------------------------------------------

IMG_SIZE = 128
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=10
)

train_gen = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="file_path",
    y_col="gender",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_gen = datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="file_path",
    y_col="gender",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

print(train_gen.class_indices)

# -------------------------------------------------------
# 6. CNN Model
# -------------------------------------------------------

model1 = Sequential([

    Conv2D(32,(3,3),activation='relu',input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPooling2D((2,2)),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D((2,2)),

    Conv2D(128,(3,3),activation='relu'),
    MaxPooling2D((2,2)),

    Conv2D(256,(3,3),activation='relu'),
    MaxPooling2D((2,2)),

    Flatten(),

    Dense(256,activation='relu'),
    Dropout(0.5),

    Dense(1,activation='sigmoid')
])

# -------------------------------------------------------
# 7. Compile Model
# -------------------------------------------------------

model1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model1.summary()

# -------------------------------------------------------
# 8. Train Model
# -------------------------------------------------------

history = model1.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15
)

# -------------------------------------------------------
# 9. Save Model
# -------------------------------------------------------

model1.save("gender_prediction_model.keras")

# -------------------------------------------------------
# 10. Prediction Example
# -------------------------------------------------------

import cv2
import numpy as np

def predict_gender(img_path):

    img = cv2.imread(img_path)
    img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
    img = img/255.0
    img = np.expand_dims(img,axis=0)

    pred = model1.predict(img)[0][0]

    if pred > 0.5:
        print("Female")
    else:
        print("Male")


# Example
# predict_gender("test.jpg")

Total Images: 23708
                                           file_path  age  gender
0  /kaggle/input/utkface-new/UTKFace/26_0_2_20170...   26       0
1  /kaggle/input/utkface-new/UTKFace/22_1_1_20170...   22       1
2  /kaggle/input/utkface-new/UTKFace/21_1_3_20170...   21       1
3  /kaggle/input/utkface-new/UTKFace/28_0_0_20170...   28       0
4  /kaggle/input/utkface-new/UTKFace/17_1_4_20170...   17       1
Found 18966 validated image filenames belonging to 2 classes.
Found 4742 validated image filenames belonging to 2 classes.
{'female': 0, 'male': 1}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,359,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,748,225 (10.48 MB)

 Trainable params: 2,748,225 (10.48 MB)

 Non-trainable params: 0 (0.00 B)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 521s 876ms/step - accuracy: 0.6677 - loss: 0.5923 - val_accuracy: 0.8307 - val_loss: 0.3697
Epoch 2/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 496s 837ms/step - accuracy: 0.8402 - loss: 0.3692 - val_accuracy: 0.8642 - val_loss: 0.3104
Epoch 3/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 495s 834ms/step - accuracy: 0.8569 - loss: 0.3238 - val_accuracy: 0.8697 - val_loss: 0.2887
Epoch 4/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 512s 862ms/step - accuracy: 0.8736 - loss: 0.2925 - val_accuracy: 0.8781 - val_loss: 0.2750
Epoch 5/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 587s 905ms/step - accuracy: 0.8793 - loss: 0.2777 - val_accuracy: 0.8825 - val_loss: 0.2641
Epoch 6/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 557s 939ms/step - accuracy: 0.8888 - loss: 0.2624 - val_accuracy: 0.8887 - val_loss: 0.2547
Epoch 7/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 492s 829ms/step - accuracy: 0.8890 - loss: 0.2533 - val_accuracy: 0.8937 - val_loss: 0.2341
Epoch 8/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 510s 859ms/step - accuracy: 0.8976 -